# Unused Enzyme Sector Parametrization
Calculation of unused enzyme fraction based on data of Utrilla et al. (2016) and Bruggeman et al. (2020)
Main assumptions:
- at mu = 0 h^(-1) the 37% of the metabolic proteins is unused (Bruggeman et al. 2020)
- all non growth proteins in ALE strain are housekeeping proteins
- all non growth proteins in WT strain are either housekeeping or unused proteins
(Although they say that mostly the hedging genes are downresulated, those are not related to growth rate)
See Figure 5C and Table S2


### 0. Setting up the notebook environment

In [4]:
#packages
import os
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import math
from PAModelpy.EnzymeSectors import UnusedEnzymeSector

if os.path.split(os.getcwd())[1] == 'Protein_sectors':
    os.chdir('../..')

from Scripts.pam_generation import setup_ecoli_pam
from Modules.utils.sector_config_functions import (perform_linear_regression, plot_unused_protein_vs_mu,
                                                            get_model_simulations_vs_sector)
#also need xlwt for to_excel function in pandas

In [5]:
#constants
#files and file paths
MODEL_DIR ='Models'
DATA_FILE_PATH = os.path.join('Data', 'Ecoli_phenotypes', 'Proteome','Schmidt-quantitative-proteomics_2010_parsed.xls')
PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls')
VALID_DATA_FILE_PATH = os.path.join('Data', 'Ecoli_phenotypes', 'Ecoli_phenotypes_py.xls')

#data parsing
PROTEIN_FRACTION = 0.258 #[g_prot/g_cdw]
MEASURED_PROTEIN_FRACTION = 0.55*0.55#0.55 * 0.55 #measured proteome vs g_p/gcdw in Ecoli *0.8 correction for proteins in model
GLC_UPTAKE_RXN_ID = 'EX_glc__D_e'
BIOMASS_RXN_ID = 'BIOMASS_Ec_iML1515_core_75p37M'

# MAX_GROWTH_ALE = 0.86 # by Utrilla et al (2016) in minimal medium or 0.98 by LaCroix et al (2015)?
MAX_GROWTH_ALE = math.log(2)*1.63#Or from lenski experiment which is ln(2)*1.63(doublings per hour)
UNUSED_PROTEIN_INTERCEPT = 0.37*MEASURED_PROTEIN_FRACTION #37% (Bruggeman et al (2020)) of all the proteins which can be measured


## 1. Setting up the model

In [15]:
pamodel = setup_ecoli_pam(pam_data_file_path = os.path.join(
    'Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),sensitivity=False)

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector



/home/samiralvdb/.local/lib/python3.10/site-packages/PAModelpy/PAModel.py:230: UserWarning:

Molar mass for E332 is invalid: 0.0



Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedProteinSector

Done with setting up the proteome allocation model iML1515



## 2. Run simulations to see how the growth rate relates to unused protein sector

In [19]:
substrate_uptake_rates = np.arange(-11, 1, 1)
ups_sector = pamodel.sectors.get_by_id('UnusedProteinSector')

print('Running simulations for the PAM with the old parameters')
simulation_results = get_model_simulations_vs_sector(pamodel,BIOMASS_RXN_ID, GLC_UPTAKE_RXN_ID, substrate_uptake_rates, 
                               ups_sector.intercept/1e3, ups_sector.slope/1e3, sector_name = 'unused_protein')
simulation_results['unused_protein'] = simulation_results['unused_protein']

Running simulations for the PAM with the old parameters
Running simulations with  -11 mmol/g_cdw/h of substrate going into the system


/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



Running simulations with  -10 mmol/g_cdw/h of substrate going into the system


/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



Running simulations with  -9 mmol/g_cdw/h of substrate going into the system


...

/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



Running simulations with  -1 mmol/g_cdw/h of substrate going into the system


/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



Running simulations with  0 mmol/g_cdw/h of substrate going into the system


/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



In [17]:
ups_molmass = ups_sector.mol_mass[0]
plot_unused_protein_vs_mu(simulation_results, ups_molmass, BIOMASS_RXN_ID)

## 3. New relation mu and unused protein

In [18]:
x = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
ue_0 = UNUSED_PROTEIN_INTERCEPT
y = [(ue_0-ue_0/MAX_GROWTH_ALE*mu) for mu in x]
print(ue_0/MAX_GROWTH_ALE)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, 
                             y=y,
                             mode='lines', line = dict(width = 5)))
fig.update_xaxes(title_text = 'Growth rate ($h^{-1}$)', range=[0,1])
fig.update_yaxes(title_text = 'Unused enzyme sector ($g_{unusedprotein}/g_{CDW}$)', range = [0,0.2])
    
fig.update_traces(marker_size=10)
fig.update_layout(font = dict(size = 12), title = 'Glucose limitation')
fig.show()

0.09906358432607194


In [11]:
#change the enzyme sector information
pamodel_new = setup_ecoli_pam(pam_data_file_path = os.path.join(
    'Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),sensitivity=False)
pamodel_new.change_sector_parameters(sector = pamodel.sectors.get_by_id('UnusedProteinSector'),
                                 slope= -ue_0/MAX_GROWTH_ALE, intercept=ue_0, lin_rxn_id= BIOMASS_RXN_ID)

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector



/home/samiralvdb/.local/lib/python3.10/site-packages/PAModelpy/PAModel.py:230: UserWarning:

Molar mass for E332 is invalid: 0.0



Add the following protein sector: TranslationalProteinSector

Add the following protein sector: UnusedProteinSector

Done with setting up the proteome allocation model iML1515



In [13]:
#run the simulations
simulation_results_new = get_model_simulations_vs_sector(pamodel_new, GLC_UPTAKE_RXN_ID, BIOMASS_RXN_ID, substrate_uptake_rates, 
                               ue_0, -ue_0/MAX_GROWTH_ALE, sector_name = 'unused_protein')

Running simulations with  -11 mmol/g_cdw/h of substrate going into the system
Running simulations with  -10 mmol/g_cdw/h of substrate going into the system
Running simulations with  -9 mmol/g_cdw/h of substrate going into the system
Running simulations with  -8 mmol/g_cdw/h of substrate going into the system
Running simulations with  -7 mmol/g_cdw/h of substrate going into the system
Running simulations with  -6 mmol/g_cdw/h of substrate going into the system
Running simulations with  -5 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -1 mmol/g_cdw/h of substrate going into the system
Running simulations with  0 mmol/g_cdw/h of substrate going into the system


/home/samiralvdb/.local/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning:

Solver status is 'infeasible'.



In [14]:
plot_unused_protein_vs_mu(simulation_results_new, ups_molmass, BIOMASS_RXN_ID)